# Reflex-1 quickstart

Frontier-LLM accuracy at Jev speed, with a 4B open model.

This notebook starts the Reflex-1 endpoint and sends four kinds of requests:
1. **Intent classification** (instant, text)
2. **Image classification** (instant, image)
3. **Reasoning with trace** (deep, latent reasoning)
4. **Free-text answer**

**Runtime:** GPU with bfloat16 support (Colab **L4** or **A100**). The free T4 does not support bfloat16.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/matu79go/reflex-1.git
%cd reflex-1
!pip -q install -r requirements.txt

Start the endpoint in the background. The first start downloads Gemma 4 E4B and the three skill adapters (a few minutes).

In [ ]:
import subprocess, time, json, base64, requests
srv = subprocess.Popen(['python', '-m', 'reflex.server', '--skills', 'skills.json', '--port', '8097', '--quant', 'nf4'],
                       stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        if requests.get('http://localhost:8097/health', timeout=2).ok:
            print('ready'); break
    except Exception:
        pass
    time.sleep(5)
else:
    print(open('server.log').read()[-2000:])

def decide(req):
    r = requests.post('http://localhost:8097/v1/decisions', json=req, timeout=600).json()
    return r

## 1. Intent classification

Route customer inquiries to a desk. Options can come with short descriptions.

In [ ]:
desks = {'credit_score': 'credit scores or ratings', 'transfer': 'moving money between accounts',
         'lost_card': 'a lost or stolen card', 'none': 'none of these'}
for text in ['I want to know my credit rating', 'send 200 dollars to my savings account',
             'I think someone stole my card', 'what is the weather tomorrow']:
    r = decide({'state': text, 'questions': {'desk': {'type': 'choice', 'instructions': 'Which desk should handle this inquiry?', 'criteria': desks}}})
    a = r['answers']['desk']
    print(f"{text!r:45} -> {a['choice']:12} ({a['confidence']:.2f}, {r['latency_ms']} ms)")

## 2. Image classification

A public-domain photo from Wikimedia Commons ([source](https://commons.wikimedia.org/wiki/File:Margherita_pizza_55.jpg), CC0).

In [ ]:
img = requests.get("https://commons.wikimedia.org/wiki/Special:FilePath/Margherita_pizza_55.jpg" + '?width=640', headers={'User-Agent': 'reflex-1-quickstart'}).content
dishes = {k: '' for k in ['pizza', 'sushi', 'ramen', 'hamburger', 'ice cream', 'steak', 'french fries', 'fried rice', 'caesar salad', 'chocolate cake']}
r = decide({'state': 'Classify the dish in the attached photo.', 'image': base64.b64encode(img).decode(),
            'questions': {'dish': {'type': 'choice', 'instructions': 'Which dish is this?', 'criteria': dishes}}})
print(r['answers']['dish']['choice'], r['answers']['dish']['confidence'], r['latency_ms'], 'ms')

## 3. Reasoning with trace (deep mode)

A 10-person chain of truth-tellers and liars. Instant models, including frontier LLMs with reasoning off, answer this at about coin-flip accuracy. Reflex-1 reasons in latent space, one step per person, and `trace` decodes each step into words.

In [ ]:
q = json.load(open('examples/deep_trace.json'))
r = decide(q)
a = r['answers']['answer']
print('answer:', a['choice'], a['confidence'], r['latency_ms'], 'ms')
print('\n'.join(a['trace']))

Change the chain and ask again. Each sentence must follow the same format: `X says Y lies.` or `X says Y tells the truth.` Use full first names like the ones below (the skill was trained on this style of names).

In [ ]:
names = ['Sherrie', 'Vernell', 'Alexis', 'Michaela', 'Delbert', 'Jerry', 'Amberly', 'Crista']
import random
random.seed(1)
parts = [f'{names[0]} tells the truth.']
for prev, cur in zip(names, names[1:]):
    parts.append(f"{cur} says {prev} {random.choice(['lies', 'tells the truth'])}.")
state = 'Question: ' + ' '.join(parts) + f' Does {names[-1]} tell the truth?'
print(state)
r = decide({'state': state, 'mode': 'deep', 'skill': 'web_of_lies', 'trace': True,
            'questions': {'answer': {'type': 'choice', 'instructions': 'Answer Yes or No.', 'criteria': {'Yes': '', 'No': ''}}}})
print(r['answers']['answer']['choice'], r['latency_ms'], 'ms')
print('\n'.join(r['answers']['answer']['trace']))

## 4. Free-text answer

In [ ]:
r = decide(json.load(open('examples/text.json')))
print(r['answers']['why']['text'])